In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
import pandas as pd
import pprint

In [22]:
from eml_transformer.ingestion.sources.iem_afos import IEMAFOSSource

source = IEMAFOSSource(
    product_types=["SPS", ],
    wfos=["IND"],
    limit=3, 
)

records = source.fetch_records(
    from_date="2024-01-01",
    to_date="2025-01-05",
)

In [23]:
print("RAW RESPONSES:", len(records))
print("RAW KEYS:", records[0].keys())

RAW RESPONSES: 3
RAW KEYS: dict_keys(['source_id', 'pil', 'raw_text', 'header', 'issued_at_text', 'published_at'])


In [24]:
records

[{'source_id': '26b08e9e0aa2244bbd83cf90743653afdb62d55de5d17915c0a7cbdf5de5fa08',
  'pil': 'SPSIND',
  'raw_text': '236 \nWWUS83 KIND 021957\nSPSIND\n\nSpecial Weather Statement\nNational Weather Service Indianapolis IN\n257 PM EST Thu Jan 2 2025\n\nINZ021-028>031-035>049-054>057-063>065-072-031000-\nCarroll-Warren-Tippecanoe-Clinton-Howard-Fountain-Montgomery-\nBoone-Tipton-Hamilton-Madison-Delaware-Randolph-Vermillion-Parke-\nPutnam-Hendricks-Marion-Hancock-Henry-Morgan-Johnson-Shelby-Rush-\nBrown-Bartholomew-Decatur-Jennings-\nIncluding the cities of Delphi, Flora, Williamsport, \nWest Lebanon, Lafayette, West Lafayette, Frankfort, Kokomo, \nAttica, Covington, Veedersburg, Crawfordsville, Lebanon, \nZionsville, Tipton, Fishers, Carmel, Noblesville, Anderson, \nMuncie, Winchester, Union City, Farmland, Parker City, Clinton, \nFairview Park, Rockville, Montezuma, Rosedale, Greencastle, \nPlainfield, Brownsburg, Danville, Indianapolis, Greenfield, \nNew Castle, Martinsville, Mooresvil

In [25]:
print("PARSED RECORDS:", len(records))

for i, record in enumerate(records):
    print("\n" + "=" * 80)
    print(f"RECORD {i + 1}")
    print("=" * 80)

    raw_text = record["raw_text"]
    header = source._parse_header(raw_text)
    sections = source._parse_sections(raw_text)
    issued = source._extract_issued_text(raw_text)

    print("PIL:", record["pil"])
    print("HEADER:", header)
    print("ISSUED:", issued)
    print("SECTIONS:", list(sections.keys()))
    print("RAW LEN:", len(raw_text))

    print("\nFIRST 10 LINES:")
    print("\n".join(raw_text.splitlines()[:10]))

    print("\nKEY MESSAGES:")
    print(sections.get("KEY MESSAGES", "MISSING")[:1000])

PARSED RECORDS: 3

RECORD 1
PIL: SPSIND
HEADER: {'raw_id': '236', 'wmo': 'WWUS83', 'wmo_header': 'WWUS83 KIND 021957', 'office': 'KIND', 'issued_code': '021957', 'pil': 'SPSIND'}
ISSUED: 257 PM EST Thu Jan 2 2025
SECTIONS: []
RAW LEN: 1704

FIRST 10 LINES:
236 
WWUS83 KIND 021957
SPSIND

Special Weather Statement
National Weather Service Indianapolis IN
257 PM EST Thu Jan 2 2025

INZ021-028>031-035>049-054>057-063>065-072-031000-
Carroll-Warren-Tippecanoe-Clinton-Howard-Fountain-Montgomery-

KEY MESSAGES:
MISSING

RECORD 2
PIL: SPSIND
HEADER: {'raw_id': '231', 'wmo': 'WWUS83', 'wmo_header': 'WWUS83 KIND 020853', 'office': 'KIND', 'issued_code': '020853', 'pil': 'SPSIND'}
ISSUED: 353 AM EST Thu Jan 2 2025
SECTIONS: []
RAW LEN: 2347

FIRST 10 LINES:
231 
WWUS83 KIND 020853
SPSIND

Special Weather Statement
National Weather Service Indianapolis IN
353 AM EST Thu Jan 2 2025

INZ021-028>031-035>049-051>057-060>065-067>072-022130-
Carroll-Warren-Tippecanoe-Clinton-Howard-Fountain-Montgomery-

In [26]:
standardized = [
    source.standardize_record(record)
    for record in records
]

for i, record in enumerate(standardized):
    print("\n" + "=" * 80)
    print(f"STANDARDIZED {i + 1}")
    print("=" * 80)

    print("record_id:", record.record_id)
    print("title:", record.title)
    print("published_at:", record.published_at)
    print("region:", record.region)
    print("categories:", record.categories)
    print("text len:", len(record.text or ""))
    print("raw sections:", list(record.metadata.get("sections", {}).keys()))

    print("\nTEXT PREVIEW:")
    print((record.text or "")[:1000])


STANDARDIZED 1
record_id: 733eeca6e0b55f9ee4d78651c29c84f2fe9f4fcb62ba6bb70bae7dfcd2d047a9
title: SPS | KIND | 257 PM EST Thu Jan 2 2025
published_at: 2025-01-02T19:57:00+00:00
region: IND
categories: ['weather', 'nws', 'iem', 'afos', 'sps']
text len: 1704
raw sections: []

TEXT PREVIEW:
236 
WWUS83 KIND 021957
SPSIND

Special Weather Statement
National Weather Service Indianapolis IN
257 PM EST Thu Jan 2 2025

INZ021-028>031-035>049-054>057-063>065-072-031000-
Carroll-Warren-Tippecanoe-Clinton-Howard-Fountain-Montgomery-
Boone-Tipton-Hamilton-Madison-Delaware-Randolph-Vermillion-Parke-
Putnam-Hendricks-Marion-Hancock-Henry-Morgan-Johnson-Shelby-Rush-
Brown-Bartholomew-Decatur-Jennings-
Including the cities of Delphi, Flora, Williamsport, 
West Lebanon, Lafayette, West Lafayette, Frankfort, Kokomo, 
Attica, Covington, Veedersburg, Crawfordsville, Lebanon, 
Zionsville, Tipton, Fishers, Carmel, Noblesville, Anderson, 
Muncie, Winchester, Union City, Farmland, Parker City, Clinton, 
Fair

### Checking Stored

In [27]:
import json

path = "../data/bronze/source=iem_afos/records.jsonl"

with open(path, "r", encoding="utf-8") as f:
    records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("records loaded:", len(records))

standardized = [
    source.standardize_record(record['raw'])
    for record in records
]

for i, record in enumerate(standardized):
    # print("\n" + "=" * 80)
    # print(f"STANDARDIZED {i + 1}")
    # print("=" * 80)

    # print("record_id:", record.record_id)
    # print("title:", record.title)
    # print("published_at:", record.published_at)
    # print("region:", record.region)
    # print("categories:", record.categories)
    # print("text len:", len(record.text or ""))

    sections = record.metadata.get("sections", {}) if record.metadata else {}
    print("raw sections:", list(sections.keys()))

    # print("\nTEXT PREVIEW:")
    # print((record.text or "")[:1000])

records loaded: 37319
raw sections: ['KEY MESSAGES', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'FORECAST UPDATE', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'FORECAST UPDATE', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'FORECAST UPDATE', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
r

In [28]:
df = pd.read_parquet('../data/silver/source=iem_afos/records.parquet')

In [29]:
df.columns

Index(['record_id', 'source', 'source_type', 'title', 'text', 'published_at',
       'retrieved_at', 'url', 'region', 'categories', 'metadata', 'raw'],
      dtype='object')

In [30]:
row = df.iloc[0]

for col in df.columns:
    print("\n" + "=" * 80)
    print(col.upper())
    print("=" * 80)
    print(row[col])


RECORD_ID
2d552b3879fb743a7d14c34a4706772fcfa3a33d44a539f5a94e4834f85b867b

SOURCE
iem_afos

SOURCE_TYPE
api

TITLE
HWO | KIWX | 254 AM EST Wed Jan 1 2025

TEXT
947 FLUS43 KIWX 010754 HWOIWX Hazardous Weather Outlook National Weather Service Northern Indiana 254 AM EST Wed Jan 1 2025 INZ005>009-012>015-017-018-020-022>027-032>034-103-104-116-203-204- 216-MIZ078>081-177-277-OHZ001-002-004-005-015-016-024-025-020800- Elkhart-Lagrange-Steuben-Noble-De Kalb-Starke-Pulaski-Marshall- Fulton IN-Whitley-Allen IN-White-Cass IN-Miami-Wabash-Huntington- Wells-Adams-Grant-Blackford-Jay-Northern La Porte- Eastern St. Joseph IN-Northern Kosciusko-Southern La Porte- Western St. Joseph IN-Southern Kosciusko-Cass MI-St. Joseph MI- Branch-Hillsdale-Northern Berrien-Southern Berrien-Williams- Fulton OH-Defiance-Henry-Paulding-Putnam-Van Wert-Allen OH- 254 AM EST Wed Jan 1 2025 /154 AM CST Wed Jan 1 2025/ This Hazardous Weather Outlook is for northern Indiana, southwest Michigan and northwest Ohio. .DAY 

In [31]:
import json

In [32]:
for i, row in df.head(5).iterrows():
    print("\n" + "#" * 100)

    print("TITLE:")
    print(row["title"])

    print("\nPUBLISHED_AT:")
    print(row["published_at"])

    print("\nREGION:")
    print(row["region"])

    print("\nTEXT PREVIEW:")
    print(row["text"][:1000])

    print("\nTEXT LENGTH:")
    print(len(row["text"]))


    print("\nSECTION KEYS:")
    print(row["metadata"].get("sections", {}).keys())


####################################################################################################
TITLE:
HWO | KIWX | 254 AM EST Wed Jan 1 2025

PUBLISHED_AT:
2025-01-01T07:54:00+00:00

REGION:
IWX

TEXT PREVIEW:
947 FLUS43 KIWX 010754 HWOIWX Hazardous Weather Outlook National Weather Service Northern Indiana 254 AM EST Wed Jan 1 2025 INZ005>009-012>015-017-018-020-022>027-032>034-103-104-116-203-204- 216-MIZ078>081-177-277-OHZ001-002-004-005-015-016-024-025-020800- Elkhart-Lagrange-Steuben-Noble-De Kalb-Starke-Pulaski-Marshall- Fulton IN-Whitley-Allen IN-White-Cass IN-Miami-Wabash-Huntington- Wells-Adams-Grant-Blackford-Jay-Northern La Porte- Eastern St. Joseph IN-Northern Kosciusko-Southern La Porte- Western St. Joseph IN-Southern Kosciusko-Cass MI-St. Joseph MI- Branch-Hillsdale-Northern Berrien-Southern Berrien-Williams- Fulton OH-Defiance-Henry-Paulding-Putnam-Van Wert-Allen OH- 254 AM EST Wed Jan 1 2025 /154 AM CST Wed Jan 1 2025/ This Hazardous Weather Outlook is for norther

In [33]:
print(df.isna().sum())

record_id       0
source          0
source_type     0
title           0
text            0
published_at    0
retrieved_at    0
url             0
region          0
categories      0
metadata        0
raw             0
dtype: int64


In [34]:
df["published_at"].head(20)

0     2025-01-01T07:54:00+00:00
1     2025-01-01T08:37:00+00:00
2     2025-01-01T09:28:00+00:00
3     2025-01-01T09:32:00+00:00
4     2025-01-01T10:00:00+00:00
5     2025-01-01T10:01:00+00:00
6     2025-01-01T11:37:00+00:00
7     2025-01-01T17:18:00+00:00
8     2025-01-01T20:20:00+00:00
9     2025-01-01T20:36:00+00:00
10    2025-01-01T20:37:00+00:00
11    2025-01-01T20:37:00+00:00
12    2025-01-01T20:42:00+00:00
13    2025-01-01T20:52:00+00:00
14    2025-01-01T20:52:00+00:00
15    2025-01-01T20:59:00+00:00
16    2025-01-01T21:06:00+00:00
17    2025-01-02T02:37:00+00:00
18    2025-01-02T02:50:00+00:00
19    2025-01-02T08:30:00+00:00
Name: published_at, dtype: object

In [35]:
df.applymap(type).head(1).T

/tmp/ipykernel_3834/4094605693.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df.applymap(type).head(1).T


,0
record_id,<class 'str'>
source,<class 'str'>
source_type,<class 'str'>
title,<class 'str'>
text,<class 'str'>
published_at,<class 'str'>
retrieved_at,<class 'str'>
url,<class 'str'>
region,<class 'str'>
categories,<class 'numpy.ndarray'>


In [36]:
df["metadata"].apply(
    lambda r: list(r.get("sections", {}).keys())
).value_counts()

metadata
[DAY ONE, DAYS TWO THROUGH SEVEN, SPOTTER INFORMATION STATEMENT, KEY MESSAGES, SHORT TERM, LONG TERM, AVIATION, LSX WATCHES/WARNINGS/ADVISORIES, DISCUSSION, AVIATION /18Z TAFS THROUGH 18Z TUESDAY/, DMX WATCHES/WARNINGS/ADVISORIES, AVIATION /00Z TAFS THROUGH 00Z WEDNESDAY/, AVIATION /06Z TAFS THROUGH 06Z WEDNESDAY/, UPDATE, SHORT TERM /THROUGH TUESDAY NIGHT/, LONG TERM /WEDNESDAY THROUGH MONDAY/, AVIATION /00Z TAFS THROUGH 00Z TUESDAY/, DVN WATCHES/WARNINGS/ADVISORIES, SHORT TERM /THROUGH TONIGHT/, AVIATION /12Z TAFS THROUGH 12Z WEDNESDAY/, AVIATION /18Z TAFS THROUGH 18Z WEDNESDAY/, AVIATION /06Z TAFS THROUGH 06Z THURSDAY/, IWX WATCHES/WARNINGS/ADVISORIES, SHORT TERM /THROUGH WEDNESDAY/, LONG TERM /WEDNESDAY NIGHT THROUGH TUESDAY/, LONG TERM /WEDNESDAY NIGHT THROUGH MONDAY/, AVIATION /00Z TAFS THROUGH 00Z THURSDAY/, AVIATION /18Z TAFS THROUGH 18Z THURSDAY/, AVIATION /12Z TAFS THROUGH 12Z THURSDAY/, AVIATION /12Z TAFS THROUGH 18Z THURSDAY/, LOT WATCHES/WARNINGS/ADVISORIES, AVIAT

In [38]:
min(df['published_at'])

'2025-01-01T07:54:00+00:00'